<a href="https://colab.research.google.com/github/hbankar11/Pyspark_Learning/blob/main/CompletePySpark_2026_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
#Just Checking PySpark is Installed or not
from pyspark.sql import SparkSession

In [3]:
# Create spark session
spark_session = SparkSession.builder.appName("CompletePySpark_2026-04").getOrCreate()

In [4]:
#Create SparkContext in PySpark
print(spark_session.sparkContext)

<SparkContext master=local[*] appName=CompletePySpark_2026-04>


In [ ]:
print("Spark App Name : " + spark_session.sparkContext.appName)

Spark App Name : CompletePySpark_2026-04


In [ ]:
#Stop PySpark SparkContext
#When PySpark executes this statement, it logs the message INFO SparkContext: Successfully stopped SparkContext to console or to a log file.
spark_session.sparkContext.stop()

In [ ]:
#Create RDD using parallelize
data = [1,2,3,4,5,6,7,8,9,10]
rdd = spark_session.sparkContext.parallelize(data)
rdd.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [ ]:
#create RDD using sparkContext.textFile()
rdd = spark_session.sparkContext.textFile("<path>")


In [ ]:
#create RDD using sparkContext.wholeTextFiles()
rdd3 = spark_session.sparkContext.wholeTextFiles("<path>")

In [ ]:
#Create empty RDD using sparkContext.emptyRDD
emptyRdd = spark_session.sparkContext.emptyRDD
print(emptyRdd)

<bound method SparkContext.emptyRDD of <SparkContext master=local[*] appName=CompletePySpark_2026-04>>


In [ ]:
#Creating empty RDD with partition
emptyRddPartition = spark_session.sparkContext.parallelize([],10)
emptyRddPartition.collect()

[]

In [ ]:
#RDD Partitions
#Get Partition count
print(f"No. of partition of rdd: ",rdd.getNumPartitions())

No. of partition of rdd:  2


In [ ]:
#Set partition Manually
rddPartition = spark_session.sparkContext.parallelize([1,2,3,4],4)
#rddPartition.collect()
rddPartition.getNumPartitions()

4

In [ ]:
#repartition and coalesce
rddPartition.repartition(6).getNumPartitions() #6
rddPartition.repartition(2).getNumPartitions() #2

rddPartition.coalesce(5).getNumPartitions() #4 Because coalesce is used only to decrease the partition
rddPartition.coalesce(2).getNumPartitions() #2

2

In [ ]:
#flatMap() : flattens all those sequences into a single RDD
rdd = spark_session.sparkContext.textFile("/content/sample.txt")
rdd.flatMap(lambda x:x.split(" ")).collect()

['1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '12',
 '23',
 '34',
 '45',
 '21',
 '27',
 '29',
 '30']

In [ ]:
#map() : It allows you to apply a function to each element of an RDD, producing a new RDD with the transformed elements.
# Add a new element with value 1 to each word
rdd_map = spark_session.sparkContext.textFile("/content/sample.txt").map(lambda x: (x,1))
rdd_map.collect()

[("('Project', 3)", 1),
 ("('Gutenberg’s', 3)", 1),
 ("('Alice’s', 1)", 1),
 ("('in', 2)", 1),
 ("('Adventures', 2)", 1),
 ("('Wonderland', 2)", 1)]

In [ ]:
#reduceByKey() : combines the values associated with each key using the provided function.
rdd_reduceByKey = spark_session.sparkContext.textFile("/content/sample.txt").reduceByKey(lambda a,b: a+b)


In [ ]:
#sortByKey() : transformation arranges the elements of an RDD based on their keys.
# Sample data: (key, value) pairs
data = [(3, "apple"), (1, "banana"), (4, "grapes"), (2, "orange")]

# Parallelize into an RDD
rdd = spark_session.sparkContext.parallelize(data)

# Sort by key
sorted_rdd = rdd.sortByKey()

# Collect results
print(sorted_rdd.collect())

[(1, 'banana'), (2, 'orange'), (3, 'apple'), (4, 'grapes')]


In [ ]:
data = [(3, "apple"), (1, "banana"), (4, "grapes"), (2, "orange")]
rdd = spark_session.sparkContext.parallelize(data)

# Parallelize into an RDD
rdd = spark_session.sparkContext.parallelize(data)
#1: count() => Returns the number of records in an RDD
print("Count : "+str(rdd.count()))
#2: first() => Returns the first record.
firstRec = rdd.first()
print("First Record : "+str(firstRec))
#3: max() => Returns maximum record
datMax = rdd.max()
print("Maximum Record : "+str(datMax))
#4: reduce() => Reduces the records to single, we can use this to count or sum.
totalWordCount = rdd.reduce(lambda a,b: (a[0]+b[0],a[1]))
print("dataReduce Record : "+str(totalWordCount[0]))
#5: take() =>  Returns the record specified as an argument.
data = rdd.take(2)
print("Element at position 2 : "+str(data))


Count : 4
First Record : (3, 'apple')
Maximum Record : (4, 'grapes')
dataReduce Record : 10
Element at position 3 : [(3, 'apple'), (1, 'banana')]


In [ ]:
## Converts RDD to DataFrame
dfRdd1 = rdd.toDF()
dfRdd1.show(truncate=False)

dfRdd1 = rdd.toDF(["id","fruit"])
dfRdd1.show(truncate=False)


+---+------+
|_1 |_2    |
+---+------+
|3  |apple |
|1  |banana|
|4  |grapes|
|2  |orange|
+---+------+

+---+------+
|id |fruit |
+---+------+
|3  |apple |
|1  |banana|
|4  |grapes|
|2  |orange|
+---+------+



In [ ]:
# using createDataFrame() - Convert DataFrame to RDD
df = spark_session.createDataFrame(rdd).toDF("id","fruit")
df.show(truncate=False)

+---+------+
|id |fruit |
+---+------+
|3  |apple |
|1  |banana|
|4  |grapes|
|2  |orange|
+---+------+



In [ ]:
# Convert DataFrame to RDD
rdd = df.rdd
rdd.collect()

[Row(id=3, fruit='apple'),
 Row(id=1, fruit='banana'),
 Row(id=4, fruit='grapes'),
 Row(id=2, fruit='orange')]

In [ ]:
#1: Create empty RDD in Pyspark
emptyRdd = spark_session.sparkContext.emptyRDD()
emptyRdd.collect()

[]

In [ ]:
#2: Alternative method to create empty RDD
emptyRdd1 = spark_session.sparkContext.parallelize([])
emptyRdd1.collect()

[]

In [ ]:
#3:Create Empty DataFrame with Schema (StructType)
from pyspark.sql.types import *
schema = StructType([
    StructField("id",StringType(),True),
    StructField("name",StringType(),True)
])
emptyRdd = spark_session.sparkContext.emptyRDD()
df = spark_session.createDataFrame(emptyRdd,schema)
df.show(truncate=False)

+---+----+
|id |name|
+---+----+
+---+----+



In [ ]:
#Convert Empty RDD to DataFrame
df = emptyRdd.toDF(schema)
df.show(truncate=False)

+---+----+
|id |name|
+---+----+
+---+----+



In [ ]:
#Create Empty DataFrame with Schema.
df = spark_session.createDataFrame([],schema)
df.show()

+---+----+
| id|name|
+---+----+
+---+----+



In [ ]:
#Create Empty DataFrame without Schema (no columns)
df = spark_session.createDataFrame([],StructType([]))
df.show()

++
||
++
++



In [ ]:
#Create pyspark RDD
dept = [("Finance",10),("Marketing",20),("Sales",30),("IT",40)]
rdd = spark_session.sparkContext.parallelize(dept)
rdd.collect()

[('Finance', 10), ('Marketing', 20), ('Sales', 30), ('IT', 40)]

In [ ]:
#Convert Pyspark rdd to dataframe
df = rdd.toDF()
df.show()

+---------+---+
|       _1| _2|
+---------+---+
|  Finance| 10|
|Marketing| 20|
|    Sales| 30|
|       IT| 40|
+---------+---+



In [ ]:
deptColumns = ["deptName","dept_id"]
df = rdd.toDF(deptColumns)
df.show(truncate=False)

+---------+-------+
|deptName |dept_id|
+---------+-------+
|Finance  |10     |
|Marketing|20     |
|Sales    |30     |
|IT       |40     |
+---------+-------+



In [ ]:
#Using Pyspark createDataFrame() function
df = spark_session.createDataFrame(rdd,deptColumns)
df.show(truncate=False)

+---------+-------+
|deptName |dept_id|
+---------+-------+
|Finance  |10     |
|Marketing|20     |
|Sales    |30     |
|IT       |40     |
+---------+-------+



In [ ]:
# Using createDataFrame() with StructType schema
deptSchema = StructType([
    StructField("dept_name",StringType(),True),
    StructField("dept_id",IntegerType(),True)
])

dept_df = spark_session.createDataFrame(rdd,deptSchema)
dept_df.show(truncate=False)


+---------+-------+
|dept_name|dept_id|
+---------+-------+
|Finance  |10     |
|Marketing|20     |
|Sales    |30     |
|IT       |40     |
+---------+-------+



In [ ]:
#Convert above pyspark dataframe to Pandas
#Step-1:
data = [("James","","Smith","36636","M",60000),
        ("Michael","Rose","","40288","M",70000),
        ("Robert","","Williams","42114","",400000),
        ("Maria","Anne","Jones","39192","F",500000),
        ("Jen","Mary","Brown","","F",0)]

columns = ["first_name","middle_name","last_name","dob","gender","salary"]

pyspark_df = spark_session.createDataFrame(data,columns)
pyspark_df.show(truncate=False)

#Step-2: Convert pyspark dataframe to pandas
pandas_df = pyspark_df.toPandas()
print(pandas_df)

+----------+-----------+---------+-----+------+------+
|first_name|middle_name|last_name|dob  |gender|salary|
+----------+-----------+---------+-----+------+------+
|James     |           |Smith    |36636|M     |60000 |
|Michael   |Rose       |         |40288|M     |70000 |
|Robert    |           |Williams |42114|      |400000|
|Maria     |Anne       |Jones    |39192|F     |500000|
|Jen       |Mary       |Brown    |     |F     |0     |
+----------+-----------+---------+-----+------+------+

  first_name middle_name last_name    dob gender  salary
0      James                 Smith  36636      M   60000
1    Michael        Rose            40288      M   70000
2     Robert              Williams  42114         400000
3      Maria        Anne     Jones  39192      F  500000
4        Jen        Mary     Brown             F       0


In [ ]:
#PySpark show() – Display DataFrame Contents in Table
#1:Default - displays 20 rows and 20 charactes from column value
columns = ["Seqno","Quote"]
data = [("1", "Be the change that you wish to see in the world"),
    ("2", "Everyone thinks of changing the world, but no one thinks of changing himself."),
    ("3", "The purpose of our lives is to be happy."),
    ("4", "Be cool.")]
df = spark_session.createDataFrame(data,columns)
df.show()

+-----+--------------------+
|Seqno|               Quote|
+-----+--------------------+
|    1|Be the change tha...|
|    2|Everyone thinks o...|
|    3|The purpose of ou...|
|    4|            Be cool.|
+-----+--------------------+



In [ ]:
#display full column contents
df.show(truncate=False)

+-----+-----------------------------------------------------------------------------+
|Seqno|Quote                                                                        |
+-----+-----------------------------------------------------------------------------+
|1    |Be the change that you wish to see in the world                              |
|2    |Everyone thinks of changing the world, but no one thinks of changing himself.|
|3    |The purpose of our lives is to be happy.                                     |
|4    |Be cool.                                                                     |
+-----+-----------------------------------------------------------------------------+



In [ ]:
# Display 2 rows and full column contents
df.show(2,truncate=False)

+-----+-----------------------------------------------------------------------------+
|Seqno|Quote                                                                        |
+-----+-----------------------------------------------------------------------------+
|1    |Be the change that you wish to see in the world                              |
|2    |Everyone thinks of changing the world, but no one thinks of changing himself.|
+-----+-----------------------------------------------------------------------------+
only showing top 2 rows


In [ ]:
# Display 2 rows & column values 25 characters
df.show(2,truncate=25)

+-----+-------------------------+
|Seqno|                    Quote|
+-----+-------------------------+
|    1|Be the change that you...|
|    2|Everyone thinks of cha...|
+-----+-------------------------+
only showing top 2 rows


In [ ]:
# Display DataFrame rows & columns vertically
df.show(n=3,truncate=25,vertical=True)

-RECORD 0--------------------------
 Seqno | 1                         
 Quote | Be the change that you... 
-RECORD 1--------------------------
 Seqno | 2                         
 Quote | Everyone thinks of cha... 
-RECORD 2--------------------------
 Seqno | 3                         
 Quote | The purpose of our liv... 
only showing top 3 rows


In [ ]:
#PySpark StructType & StructField Explained with Examples
#Using PySpark StructType & StructField with DataFrame
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
data = [("James","","Smith",36636,"M",3000),
    ("Michael","Rose","",40288,"M",4000),
    ("Robert","","Williams",42114,"M",4000),
    ("Maria","Anne","Jones",39192,"F",4000),
    ("Jen","Mary","Brown",None,"F",-1)
  ]

schema = StructType([
    StructField("First_Name",StringType(),True),
    StructField("Middle_Name",StringType(),True),
    StructField("Last_Name",StringType(),True),
    StructField("ID",IntegerType(),True),
    StructField("Gender",StringType(),True),
    StructField("Salary",StringType(),True)
])

df = spark_session.createDataFrame(data,schema)
df.show(truncate=False)


+----------+-----------+---------+-----+------+------+
|First_Name|Middle_Name|Last_Name|ID   |Gender|Salary|
+----------+-----------+---------+-----+------+------+
|James     |           |Smith    |36636|M     |3000  |
|Michael   |Rose       |         |40288|M     |4000  |
|Robert    |           |Williams |42114|M     |4000  |
|Maria     |Anne       |Jones    |39192|F     |4000  |
|Jen       |Mary       |Brown    |NULL |F     |-1    |
+----------+-----------+---------+-----+------+------+



In [6]:
#nested structure:
from pyspark.sql.types import *
structureData = [
    (("James","","Smith"),"36636","M",3100),
    (("Michael","Rose",""),"40288","M",4300),
    (("Robert","","Williams"),"42114","M",1400),
    (("Maria","Anne","Jones"),"39192","F",5500),
    (("Jen","Mary","Brown"),"","F",-1)
  ]

nested_schema = StructType([
    StructField("name",StructType([
        StructField("first_name",StringType(),True),
        StructField("middle_name",StringType(),True),
        StructField("last_name",StringType(),True)
    ]),True),
    StructField("id",StringType(),True),
    StructField("gender",StringType(),True),
    StructField("salary",IntegerType(),True)
])

df = spark_session.createDataFrame(structureData,nested_schema)
df.show(truncate=False)
'''
+--------------------+-----+------+------+
|name                |id   |gender|salary|
+--------------------+-----+------+------+
|{James, , Smith}    |36636|M     |3100  |
|{Michael, Rose, }   |40288|M     |4300  |
|{Robert, , Williams}|42114|M     |1400  |
|{Maria, Anne, Jones}|39192|F     |5500  |
|{Jen, Mary, Brown}  |     |F     |-1    |
+--------------------+-----+------+------+
'''

+--------------------+-----+------+------+
|name                |id   |gender|salary|
+--------------------+-----+------+------+
|{James, , Smith}    |36636|M     |3100  |
|{Michael, Rose, }   |40288|M     |4300  |
|{Robert, , Williams}|42114|M     |1400  |
|{Maria, Anne, Jones}|39192|F     |5500  |
|{Jen, Mary, Brown}  |     |F     |-1    |
+--------------------+-----+------+------+



'\n+--------------------+-----+------+------+\n|name                |id   |gender|salary|\n+--------------------+-----+------+------+\n|{James, , Smith}    |36636|M     |3100  |\n|{Michael, Rose, }   |40288|M     |4300  |\n|{Robert, , Williams}|42114|M     |1400  |\n|{Maria, Anne, Jones}|39192|F     |5500  |\n|{Jen, Mary, Brown}  |     |F     |-1    |\n+--------------------+-----+------+------+\n'

In [7]:
#Updating existing structtype using struct
from pyspark.sql.functions import *
updated_df = df.withColumn("OtherInfo",
                           struct(col("id").alias("identifier"),
                                  col("gender").alias("gender"),
                                  col("salary").alias("salary"),
                                  when(col("salary").cast(IntegerType()) < 2000,"low") \
                                 .when(col("salary").cast(IntegerType()) < 4000,"Medium") \
                                 .otherwise("High").alias("Salary_Grade"))).drop("id","gender","salary")

updated_df.show(truncate=False)
'''
+--------------------+------------------------+
|name                |OtherInfo               |
+--------------------+------------------------+
|{James, , Smith}    |{36636, M, 3100, Medium}|
|{Michael, Rose, }   |{40288, M, 4300, High}  |
|{Robert, , Williams}|{42114, M, 1400, low}   |
|{Maria, Anne, Jones}|{39192, F, 5500, High}  |
|{Jen, Mary, Brown}  |{, F, -1, low}          |
+--------------------+------------------------+
'''

+--------------------+------------------------+
|name                |OtherInfo               |
+--------------------+------------------------+
|{James, , Smith}    |{36636, M, 3100, Medium}|
|{Michael, Rose, }   |{40288, M, 4300, High}  |
|{Robert, , Williams}|{42114, M, 1400, low}   |
|{Maria, Anne, Jones}|{39192, F, 5500, High}  |
|{Jen, Mary, Brown}  |{, F, -1, low}          |
+--------------------+------------------------+



In [8]:
arrayStructureSchema = StructType([
    StructField('name', StructType([
       StructField('firstname', StringType(), True),
       StructField('middlename', StringType(), True),
       StructField('lastname', StringType(), True)
       ])),
       StructField('hobbies', ArrayType(StringType()), True),
       StructField('properties', MapType(StringType(),StringType()), True)
    ])


In [11]:
#Check if column exists in the dataframe

if 'firstname' in df.columns:
  print("firstname is present in the Dataframe")
else:
  print("firstname is not present in the Dataframe")

firstname is not present in the Dataframe


In [19]:
column_name = 'id'
if column_name  in [field.name for field in df.schema]:
  print(f"'{column_name}' is present in the DataFrame schema.")
else:
    print(f"'{column_name}' is not present in the DataFrame schema.")

'id' is present in the DataFrame schema.


In [20]:
#Create column class object:
from pyspark.sql.functions import lit
colObj = lit("HemanTejal")

In [21]:
#Accessing columns from dataframe
data=[("James",23),("Ann",40)]
df=spark_session.createDataFrame(data).toDF("name.fname","gender")
df.printSchema()
'''
root
 |-- name.fname: string (nullable = true)
 |-- gender: long (nullable = true)
'''


root
 |-- name.fname: string (nullable = true)
 |-- gender: long (nullable = true)



In [22]:
#1:# Using DataFrame object (df)
df.select(df.gender).show(truncate=False)
df.select(df["gender"],df["`name.fname`"]).show(truncate=False)
#2: Using col() function
df.select(col("gender"),col("`name.fname`")).show(truncate=False)
df.select(col("gender")).show(truncate=False)


+------+
|gender|
+------+
|23    |
|40    |
+------+

+------+----------+
|gender|name.fname|
+------+----------+
|23    |James     |
|40    |Ann       |
+------+----------+



In [24]:
#Select all Columns
df.select(col("*")).show(truncate=False)
df.select("*").show(truncate=False)

+----------+------+
|name.fname|gender|
+----------+------+
|James     |23    |
|Ann       |40    |
+----------+------+



In [ ]:
#Create DataFrame with struct using Row class
